# Bài học 10 - Đại lý AI trong môi trường sản xuất

Trong bài học này bạn sẽ học **các mẫu triển khai** cho đại lý AI sử dụng Microsoft Agent Framework (Python). Nội dung bao gồm:

- **Khả năng quan sát** — thêm đo thời gian và ghi nhật ký vào tương tác của đại lý
- **Đánh giá** — sử dụng một đại lý đánh giá để chấm điểm chất lượng phản hồi
- **Quản lý chi phí** — các chiến lược tối ưu hóa token và lựa chọn mô hình

Tình huống là một **đại lý du lịch** giúp người dùng lên kế hoạch chuyến đi, với giám sát và đánh giá được thêm vào.


## Thiết lập


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Cân nhắc khi đưa vào sản xuất

Việc chuyển các tác tử AI từ nguyên mẫu sang môi trường sản xuất đòi hỏi chú ý kỹ lưỡng tới ba trụ cột:

1. **Khả năng quan sát** — Bạn cần có tầm nhìn về những gì tác tử đang làm, mất bao lâu và những công cụ nào nó gọi. Không có theo dõi và ghi nhật ký, việc gỡ lỗi các vấn đề trên môi trường sản xuất gần như không thể.

2. **Đánh giá** — Các kiểm tra chất lượng tự động đảm bảo các phản hồi của tác tử vẫn chính xác, đầy đủ và hữu ích theo thời gian. Một tác tử đánh giá có thể chấm điểm các phản hồi theo các tiêu chí đã được định nghĩa.

3. **Quản lý chi phí** — Việc sử dụng token trực tiếp ảnh hưởng đến chi phí. Các chiến lược như tối ưu hóa lời nhắc, lựa chọn mô hình và bộ nhớ đệm giúp giữ chi phí trong tầm kiểm soát mà không làm giảm chất lượng.


## Xây dựng một agent có thể quan sát

Chúng tôi định nghĩa các công cụ du lịch và bọc cuộc gọi của agent bằng việc đo thời gian để chúng ta có thể giám sát độ trễ. Trong môi trường sản xuất, bạn sẽ tích hợp với OpenTelemetry hoặc một backend truy vết tương tự.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Các Mẫu Đánh Giá

Một mô hình triển khai phổ biến là sử dụng một tác nhân thứ hai làm **bộ đánh giá**. Bộ đánh giá chấm điểm phản hồi của tác nhân chính dựa trên các tiêu chí được xác định trước như tính đầy đủ, độ chính xác và tính hữu ích.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Chiến lược quản lý chi phí

Kiểm soát chi phí là điều then chốt đối với các tác tử AI trong môi trường sản xuất. Dưới đây là các chiến lược chính:

| Strategy | Description |
|---|---|
| **Tối ưu hóa lời nhắc** | Giữ các hướng dẫn hệ thống ngắn gọn. Loại bỏ ngữ cảnh thừa để giảm số lượng token đầu vào. |
| **Lựa chọn mô hình** | Sử dụng các mô hình nhỏ hơn, rẻ hơn (ví dụ GPT-4o-mini) cho các tác vụ đơn giản như phân loại hoặc trích xuất, và chỉ dùng các mô hình lớn hơn cho suy luận phức tạp. |
| **Bộ nhớ đệm** | Lưu vào bộ nhớ đệm kết quả công cụ và các truy vấn thường xuyên để tránh các cuộc gọi API không cần thiết. |
| **Ngân sách token** | Đặt giới hạn `max_tokens` để tránh các phản hồi dài ngoài mong đợi. |
| **Gộp lô** | Gộp nhiều truy vấn của người dùng thành một cuộc gọi API khi có thể. |

Trong thực tế, một phương pháp phân tầng hoạt động hiệu quả: điều hướng các yêu cầu đơn giản đến một mô hình nhanh, giá rẻ và chỉ chuyển các truy vấn phức tạp hơn đến một mô hình mạnh hơn.


## Summary

Trong bài học này bạn đã học cách:

1. **Thêm khả năng quan sát** vào các tương tác của tác nhân với thông tin thời gian và ghi nhật ký, đặt nền tảng cho truy vết và giám sát.
2. **Đánh giá phản hồi của tác nhân** tự động bằng một tác nhân đánh giá chấm điểm mức độ hoàn chỉnh, chính xác và hữu ích.
3. **Quản lý chi phí** thông qua tối ưu hóa lời nhắc, lựa chọn mô hình, bộ nhớ đệm, và ngân sách token.

Những mẫu triển khai này giúp đảm bảo các tác nhân AI của bạn đáng tin cậy, có thể đo lường được và có hiệu quả chi phí khi triển khai ở quy mô lớn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi luôn cố gắng đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể có lỗi hoặc không chính xác. Văn bản gốc bằng ngôn ngữ gốc nên được coi là nguồn chính thức. Đối với những thông tin quan trọng, nên sử dụng bản dịch chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
